# Worksheet 2, Part 2 — Observation details table
### HE 1002-0755 (r-II star)

In [ ]:
from pylab import *
import warnings; warnings.filterwarnings("ignore")
from astropy.io import fits
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation
import astropy.units as u
import json, os

DATA = "../../../data"
BLUE = f"{DATA}/he1002-0755blue_multi_ut090727.fits"
RED  = f"{DATA}/he1002-0755ared_multi_ut090205_07slit.fits"
os.makedirs("results", exist_ok=True)

## 1. Read the headers

In [ ]:
KEYS = ["OBJECT","UT-DATE","UT-START","EXPTIME","RA","DEC","RA-D","DEC-D","EPOCH",
        "AIRMASS","SITENAME","SITELAT","SITELONG","SITEALT","TELESCOP","INSTRUME",
        "SLITSIZE","BINNING","OBSERVER"]

hdr = {}
for tag, path in [("blue", BLUE), ("red", RED)]:
    h = fits.open(path)[0].header
    hdr[tag] = h
    print(f"===== {tag.upper()}  ({os.path.basename(path)}) =====")
    print(f"  data shape = {fits.open(path)[0].data.shape}   (bands, orders, pixels)")
    for k in KEYS:
        if k in h:
            print(f"   {k:<9s} = {h[k]}")
    dop = [k for k in h if "DOPCOR" in k]
    print(f"   DOPCOR    : {({k: h[k] for k in dop} or 'NONE -> geocentric, not RV-corrected')}")
    print()

## 1b. The BANDID cards: which data plane is which

In [ ]:
BAND_USERS = {
    "object spectrum":
        "SMHR loads this as flux; our raw per-exposure sheets plot it",
    "noise spectrum":
        "SMHR converts to inverse variance",
    "signal-to-noise spectrum":
        "our per-order S/N cuts and the S/N labels",
    "object spectrum divided by normed flat":
        "our merge, coadd and velocities -- flat-divided, so most of the blaze is removed",
}

from hashlib import md5

for tag, path in [("blue", BLUE), ("red", RED)]:
    h = hdr[tag]
    cards = sorted([k for k in h if k.startswith("BANDID")], key=lambda x: int(x[6:]))
    print(f"===== {tag.upper()}  ({os.path.basename(path)}) =====")
    print(f"  {'plane':<8}{'card':<10}{'contents':<40}{'used for'}")
    for k in cards:
        # data planes are zero-indexed; BANDIDn is plane n-1
        print(f"  data[{int(k[6:])-1}]  {k:<10}{h[k]:<40}{BAND_USERS.get(h[k].strip(), '')}")
    sig = md5(";".join([h[k] for k in cards]).encode("utf-8")).hexdigest()
    print(f"  md5 of the joined BANDID strings: {sig}")
    print(f"     -> {'matches CarPy MIKE' if sig == '0da149208a3c8ba608226544605ed600' else 'NOT a format we recognise'}")
    print()

# How much blaze each science band still carries, across one whole order.
print("Blaze still present in each science band (one order, blue):")
cube = fits.open(BLUE)[0].data
names = {h.strip(): int(k[6:]) - 1 for k, h in
         [(k, hdr["blue"][k]) for k in hdr["blue"] if k.startswith("BANDID")]}
# use the order that carries Ca II K, so the number means something
import sys; sys.path.insert(0, "../../../scripts")
from mike_multispec import read_multispec
_o, _ = read_multispec(BLUE, band="object spectrum")
_lam = 3933.66*(1 + 215.0/299792.458)
iord = max((i for i, x in enumerate(_o) if x[0][0] < _lam < x[0][-1]),
           key=lambda i: np.nanmedian(_o[i][2]))
print(f"   (order {iord}, {_o[iord][0][0]:.0f}-{_o[iord][0][-1]:.0f} A, the one holding Ca II K)")
for nm in ("object spectrum", "flat spectrum", "object spectrum divided by normed flat"):
    a = cube[names[nm], iord]
    lo, hi = np.nanpercentile(a[np.isfinite(a)], [5, 95])
    print(f"   {nm:<40} max/min across the order = {hi/lo:6.1f}")
print("\n   The flat carries the blaze, so dividing by it removes most of it. That is why")
print("   'object / normed flat' looks far flatter than the plain object spectrum.")

## 2. A discrepancy worth flagging

In [ ]:
lco = EarthLocation.from_geodetic(lon=hdr["blue"]["SITELONG"]*u.deg,
                                  lat=hdr["blue"]["SITELAT"]*u.deg,
                                  height=hdr["blue"]["SITEALT"]*u.m)
sc  = SkyCoord(ra=hdr["blue"]["RA-D"]*u.deg, dec=hdr["blue"]["DEC-D"]*u.deg)
ut  = hdr["blue"]["UT-START"]

helio = {}
for iso in [f"2009-02-05T{ut}", f"2009-07-27T{ut}"]:
    t = Time(iso, scale="utc", location=lco)
    helio[iso[:10]] = sc.radial_velocity_correction("heliocentric", obstime=t).to(u.km/u.s).value

for d, v in helio.items():
    print(f"  {d}  ->  heliocentric correction = {v:+7.3f} km/s")
print(f"\n  difference = {abs(helio['2009-02-05']-helio['2009-07-27']):.2f} km/s"
      "  <- must resolve the date before quoting a heliocentric RV")

## 3. SIMBAD, Gaia DR3 and IRSA DUST

In [ ]:
# SIMBAD:  https://simbad.unistra.fr/simbad/sim-script
#   query coo 10 05 03.3 -08 10 02.1 radius=10s
simbad = dict(main_id="UCAC4 410-051009",
              ids=["2MASS J10050324-0810003", "SMSS J100503.24-081000.2"],
              otype="Chemically Peculiar Star",
              ra_deg=151.263518300, dec_deg=-8.166765300,
              B=15.035, V=13.687, G=None, J=11.320, H=10.691, K=10.557,
              rv_kms=222.81)

# Gaia DR3 (TAP sync, gaiadr3.gaia_source, 10" cone)
gaia = dict(source_id=3772156083998180864,
            parallax_mas=0.0611, parallax_error_mas=0.0192,
            pmra=0.3914, pmdec=-2.8113,
            G=13.174563, BP=13.917316, RP=12.338186,
            rv_kms=226.676, rv_error_kms=3.710,
            rv_nb_transits=8, rv_amplitude_robust=22.591,
            rv_chisq_pvalue=0.025431, non_single_star=0,
            teff_gspphot=5185.2)

# IRSA DUST, regSize=5 (as the worksheet specifies)
dust = dict(ebv_sandf=0.0446, ebv_sfd=0.0519)

print(f"  SIMBAD  {simbad['main_id']}  ({simbad['otype']})")
print(f"          B={simbad['B']}  V={simbad['V']}  J={simbad['J']}  H={simbad['H']}  K={simbad['K']}")
print(f"  Gaia    source_id {gaia['source_id']}   G={gaia['G']:.3f}")
print(f"          parallax {gaia['parallax_mas']:.4f} +/- {gaia['parallax_error_mas']:.4f} mas"
      f"   -> ~{1.0/gaia['parallax_mas']:.1f} kpc (naive 1/plx)")
print(f"  E(B-V)  {dust['ebv_sandf']:.4f} (Schlafly & Finkbeiner)   {dust['ebv_sfd']:.4f} (SFD98)")

## 4. Is HE 1002-0755 a spectroscopic binary?

In [ ]:
print(f"  Gaia RV      {gaia['rv_kms']:+.2f} +/- {gaia['rv_error_kms']:.2f} km/s "
      f"({gaia['rv_nb_transits']} transits)")
print(f"  SIMBAD RV    {simbad['rv_kms']:+.2f} km/s")
print(f"  difference   {gaia['rv_kms']-simbad['rv_kms']:+.2f} km/s")
print(f"  robust amp   {gaia['rv_amplitude_robust']:.1f} km/s,  p(const) = {gaia['rv_chisq_pvalue']:.3f},"
      f"  non_single_star = {gaia['non_single_star']}")

## 5. The observation details table

In [ ]:
rows = [
 ("Star name",                "HE 1002-0755"),
 ("SIMBAD identifier",        simbad["main_id"]),
 ("Object type",              simbad["otype"]),
 ("R.A. (sexagesimal)",       hdr["blue"]["RA"].strip()),
 ("R.A. (degrees)",           f"{hdr['blue']['RA-D']:.6f}"),
 ("Declination (sexagesimal)",hdr["blue"]["DEC"].strip()),
 ("Dec (degrees)",            f"{hdr['blue']['DEC-D']:.6f}"),
 ("B mag",                    simbad["B"]),
 ("V mag",                    simbad["V"]),
 ("g mag",                    "-- (no SDSS/SkyMapper g in SIMBAD)"),
 ("Gaia G / BP / RP",         f"{gaia['G']:.3f} / {gaia['BP']:.3f} / {gaia['RP']:.3f}"),
 ("J / H / K",                f"{simbad['J']} / {simbad['H']} / {simbad['K']}"),
 ("Reddening E(B-V)",         f"{dust['ebv_sandf']:.4f}  (S&F; SFD98 = {dust['ebv_sfd']:.4f})"),
 ("Bolometric correction BC0","Worksheet 4"),
 ("Observing date (UT)",      f"{hdr['blue']['UT-DATE']} {hdr['blue']['UT-START']}  (blue filename says 2009-07-27 -- unresolved)"),
 ("Exposure time",            f"{hdr['blue']['EXPTIME']:.0f} s (blue), {hdr['red']['EXPTIME']:.0f} s (red)"),
 ("Slit width",               f"{hdr['blue']['SLITSIZE']}\""),
 ("Binning",                  hdr["blue"]["BINNING"]),
 ("Airmass",                  hdr["blue"]["AIRMASS"]),
 ("Telescope / instrument",   f"{hdr['blue']['TELESCOP']} / MIKE"),
 ("Observers",                hdr["blue"]["OBSERVER"]),
 ("Heliocentric correction",  f"{helio['2009-02-05']:+.3f} km/s (if 2009-02-05); {helio['2009-07-27']:+.3f} km/s (if 2009-07-27)"),
 ("Radial velocity",          "to be measured -- needs the merged rest-frame spectrum"),
 ("S/N at 4500/5200/6500",    "Worksheet 6"),
]

w = max(len(k) for k, _ in rows)
lines = [f"OBSERVATION DETAILS TABLE -- HE 1002-0755", "="*(w+60)]
for k, v in rows:
    lines.append(f"{k:<{w}s} : {v}")
txt = "\n".join(lines)
print(txt)

open("results/part2_observation_details.txt", "w").write(txt + "\n")
json.dump(dict(header_blue={k: str(hdr["blue"][k]) for k in KEYS if k in hdr["blue"]},
               header_red={k: str(hdr["red"][k]) for k in KEYS if k in hdr["red"]},
               heliocentric_correction_kms=helio, simbad=simbad, gaia=gaia, dust=dust),
          open("results/part2_observation_details.json", "w"), indent=2)
print("\n  wrote results/part2_observation_details.{txt,json}")